# 2×2 Tables & Effect Measures — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/two-by-two-and-effect-measures.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/two-by-two-and-effect-measures.ipynb)

> Companion to the **2×2 Tables & Effect Measures** article. Take one 2×2 table (`a=15, b=85, c=30, d=70`) and compute the whole family — **RR, OR, RD, NNT, IRR** — each with a 95% confidence interval, *watching every number appear*.

## What you'll build

1. A function that turns the four cells **a, b, c, d** into **RR, OR, RD, NNT** with 95% CIs.
2. A second function for the **incidence rate ratio (IRR)** from person-time.
3. A run that reproduces the article's worked numbers **exactly** — pure `numpy` + `math`, **no API key**, no plotting libraries.

### Setup (nothing to install in Colab — `numpy` ships by default)

In [1]:
# Pure numpy + math. No scipy, no matplotlib, no API key.
import numpy as np
import math

Z = 1.96   # the 95% two-sided multiplier (we use the classic constant, not scipy)
print("ready — Z =", Z)

ready — Z = 1.96


## The table we'll use

Every comparison collapses to four counts. Ours is a trial: **15 of 100** treated patients had the event, **30 of 100** controls did.

```
                 Event   No event
   Treatment       a=15    b=85      -> 100
   Control         c=30    d=70      -> 100
```

**Risk** divides events by *everyone* (`a/(a+b)`); **odds** divides events by *non-events* (`a/b`). They agree when events are rare and diverge when events are common — the whole reason OR and RR will disagree below.

In [2]:
a, b, c, d = 15, 85, 30, 70
n_t, n_c = a + b, c + d
risk_t = a / n_t
risk_c = c / n_c
print(f"Risk (treatment) = {a}/{n_t} = {risk_t:.2f}")
print(f"Risk (control)   = {c}/{n_c} = {risk_c:.2f}")
print(f"Odds (treatment) = {a}/{b} = {a/b:.4f}")
print(f"Odds (control)   = {c}/{d} = {c/d:.4f}")

Risk (treatment) = 15/100 = 0.15
Risk (control)   = 30/100 = 0.30
Odds (treatment) = 15/85 = 0.1765
Odds (control)   = 30/70 = 0.4286


## The one function: four cells in, every measure out

For each **ratio** (RR, OR) we build the CI on the **log scale** — that is where the sampling distribution is symmetric — then exponentiate the endpoints back. **RD** is a difference, so it stays on the natural scale. A tiny **Haldane–Anscombe** fix (add 0.5 to every cell) keeps things finite if any cell is zero.

| Measure | Formula | Log-scale SE |
|---|---|---|
| RR | `risk_t / risk_c` | `sqrt(1/a + 1/c − 1/n_t − 1/n_c)` |
| OR | `ad / bc` | `sqrt(1/a + 1/b + 1/c + 1/d)` |
| RD | `risk_t − risk_c` | `sqrt(R_t(1−R_t)/n_t + R_c(1−R_c)/n_c)` (natural scale) |
| NNT | `1/|RD|`, round **up** | — |

In [3]:
def effect_measures(a, b, c, d, correction=0.5):
    """RR, OR, RD, NNT (+95% CIs) from a 2x2 table. Ratio CIs live on the log scale."""
    if min(a, b, c, d) == 0:                       # Haldane-Anscombe: avoid divide-by-zero
        a, b, c, d = a + correction, b + correction, c + correction, d + correction
    n_t, n_c = a + b, c + d
    risk_t, risk_c = a / n_t, c / n_c

    # ---- Relative risk (log scale) ----
    rr = risk_t / risk_c
    se_lnrr = np.sqrt(1/a + 1/c - 1/n_t - 1/n_c)
    rr_lo, rr_hi = np.exp(np.log(rr) - Z*se_lnrr), np.exp(np.log(rr) + Z*se_lnrr)

    # ---- Odds ratio (log scale) ----
    orr = (a * d) / (b * c)
    se_lnor = np.sqrt(1/a + 1/b + 1/c + 1/d)
    or_lo, or_hi = np.exp(np.log(orr) - Z*se_lnor), np.exp(np.log(orr) + Z*se_lnor)

    # ---- Risk difference (natural scale) and NNT ----
    rd = risk_t - risk_c
    se_rd = np.sqrt(risk_t*(1-risk_t)/n_t + risk_c*(1-risk_c)/n_c)
    rd_lo, rd_hi = rd - Z*se_rd, rd + Z*se_rd
    nnt = int(math.ceil(1/abs(rd)))                # round UP: you can't treat a fraction

    return {"RR": (rr, rr_lo, rr_hi), "OR": (orr, or_lo, or_hi),
            "RD": (rd, rd_lo, rd_hi), "NNT": nnt}

print("effect_measures defined")

effect_measures defined


### Run it — watch each measure and its CI print

In [4]:
r = effect_measures(a=15, b=85, c=30, d=70)
for k in ("RR", "OR", "RD"):
    est, lo, hi = r[k]
    print(f"{k:3s} = {est:5.3f}  (95% CI {lo:.3f} to {hi:.3f})")
print(f"NNT = {r['NNT']}  (treat this many to prevent one event)")

RR  = 0.500  (95% CI 0.287 to 0.870)
OR  = 0.412  (95% CI 0.205 to 0.826)
RD  = -0.150  (95% CI -0.264 to -0.036)
NNT = 7  (treat this many to prevent one event)


## Why OR looks bigger than RR

RR = 0.50 says the treatment **halves** the risk. OR = 0.41 *looks* like a bigger effect — but that is the classic **exaggeration on a common outcome** (a 15–30% event is common). A journalist quoting `1 − 0.41 = 59%` as a *risk* reduction would overstate the true **50%**. Report the RR to a clinician; the OR is the regression/meta-analysis currency.

In [5]:
rr = r['RR'][0]
orr = r['OR'][0]
print(f"RR = {rr:.3f}  -> risk reduction = {(1-rr)*100:.0f}%  (the honest number)")
print(f"OR = {orr:.3f}  -> odds reduction = {(1-orr)*100:.0f}%  (NOT a risk reduction!)")
print("OR is further from 1 than RR because the event is common.")

RR = 0.500  -> risk reduction = 50%  (the honest number)
OR = 0.412  -> odds reduction = 59%  (NOT a risk reduction!)
OR is further from 1 than RR because the event is common.


## IRR — when follow-up time varies

Headcount denominators are unfair when people are followed for different lengths of time. The **incidence rate ratio** swaps them for **person-time**: 20 events in 950 statin person-years vs 35 events in 900 placebo person-years. Its log-scale SE is `sqrt(1/E_t + 1/E_c)`.

In [6]:
def incidence_rate_ratio(e_t, pt_t, e_c, pt_c):
    """IRR with a log-scale CI; denominators are person-time, not headcount."""
    irr = (e_t / pt_t) / (e_c / pt_c)
    se = np.sqrt(1/e_t + 1/e_c)                    # SE of ln(IRR)
    return irr, np.exp(np.log(irr) - Z*se), np.exp(np.log(irr) + Z*se)

irr, lo, hi = incidence_rate_ratio(20, 950, 35, 900)
print(f"IRR = {irr:.3f}  (95% CI {lo:.3f} to {hi:.3f})")

IRR = 0.541  (95% CI 0.313 to 0.938)


## Full readout — the article's table, reproduced

One print block, all five measures. Compare against the article's *Expected output*.

In [7]:
irr, irr_lo, irr_hi = incidence_rate_ratio(20, 950, 35, 900)
for k in ('RR', 'OR', 'RD'):
    est, lo, hi = r[k]
    print(f"{k:3s} = {est:5.3f}  (95% CI {lo:.3f} to {hi:.3f})")
print(f"NNT = {r['NNT']}  (treat this many to prevent one event)")
print(f"IRR = {irr:.3f}  (95% CI {irr_lo:.3f} to {irr_hi:.3f})")

print("\n--- article Expected output ---")
print("RR  = 0.500  (95% CI 0.287 to 0.870)")
print("OR  = 0.412  (95% CI 0.205 to 0.826)")
print("RD  = -0.150 (95% CI -0.264 to -0.036)")
print("NNT = 7  (treat this many to prevent one event)")
print("IRR = 0.541  (95% CI 0.313 to 0.938)")

RR  = 0.500  (95% CI 0.287 to 0.870)
OR  = 0.412  (95% CI 0.205 to 0.826)
RD  = -0.150  (95% CI -0.264 to -0.036)
NNT = 7  (treat this many to prevent one event)
IRR = 0.541  (95% CI 0.313 to 0.938)

--- article Expected output ---
RR  = 0.500  (95% CI 0.287 to 0.870)
OR  = 0.412  (95% CI 0.205 to 0.826)
RD  = -0.150 (95% CI -0.264 to -0.036)
NNT = 7  (treat this many to prevent one event)
IRR = 0.541  (95% CI 0.313 to 0.938)


## Recap

- Every comparison is four counts **a, b, c, d**; every measure is a way of dividing or subtracting them.
- **RR = 0.50** (clinician's number) vs **OR = 0.41** (regression/meta-analysis number) — OR **overstates** on common outcomes.
- **RD = −0.15** keeps the baseline visible; **NNT = 7** is the workload per event prevented (round up, state the time horizon).
- **IRR = 0.54** uses person-time when follow-up varies.
- Build every **ratio** CI on the **log scale**, then exponentiate; **RD** stays natural.

← Back to the **2×2 Tables & Effect Measures** article for the full explanation, diagrams, and interview Q&A. Next: **Meta-Analysis: Pooling Studies**.